# HalluScan — DeBERTa Finetuning
Run this on **Colab with T4 GPU**. Takes ~45 minutes.
After training, download the `finetuned_model/` folder and put it in your repo root.

In [ ]:
!pip install transformers datasets torch accelerate -q

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load HaluEval QA subset
halueval = load_dataset('pminervini/HaluEval', 'qa_samples', split='data')
print(halueval[0])

In [ ]:
# Reformat to NLI structure: source + claim → label
# Label: 0 = Entailment (faithful), 1 = Neutral, 2 = Contradiction (hallucinated)

records = []

for item in halueval:
    source = item['knowledge']
    
    # Hallucinated answer → contradiction
    records.append({
        'source': source,
        'claim': item['hallucinated_answer'],
        'label': 2
    })
    
    # Right answer → entailment
    records.append({
        'source': source,
        'claim': item['right_answer'],
        'label': 0
    })

df = pd.DataFrame(records)
print(df.shape)
print(df['label'].value_counts())

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'cross-encoder/nli-deberta-v3-small'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['source'],
        batch['claim'],
        truncation=True,
        max_length=512,
        padding='max_length'
    )

# Train/val split
dataset = Dataset.from_pandas(df)
split = dataset.train_test_split(test_size=0.1, seed=42)
tokenized = split.map(tokenize, batched=True)
print('Train:', len(tokenized['train']), '| Val:', len(tokenized['test']))

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {'f1': f1_score(labels, preds, average='weighted')}

args = TrainingArguments(
    output_dir='./finetuned_model',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=True,
    logging_steps=50,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
# Save model — download this folder and put it in your repo as finetuned_model/
trainer.save_model('./finetuned_model')
tokenizer.save_pretrained('./finetuned_model')
print('Saved. Download the finetuned_model/ folder from the Colab file panel.')